In [1]:
import torch
import vae_model 
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "STIXGeneral",
    "text.latex.preamble": r"\usepackage[utf8]{inputenc}\usepackage{amsmath}",
    "figure.figsize": [12, 4],  # ancho, Largo  
    "xtick.labelsize": 12,  # tamaño ticks en eje x
    "ytick.labelsize": 12   # tamaño ticks en eje y
})


Generar puntos para entrenar el encoder y decoder

In [2]:
bounds = torch.tensor([[1 , 20] ,
                       [60, 60]],dtype = torch.float64)
num_initial_points = 5000
initial_points = vae_model.data_encoder_generation(bounds , num_initial_points=num_initial_points)
print('Number of points generated:', len(initial_points))
# initial_points


Number of points generated: 20000


Load Model and updated the weights of the model with the file already saved and optimized.

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = vae_model.VariationalAutoencoder(emb_dim=8,   latent_dim= 8 ).to(device)
# The .pth file should be always in the same direction where the Jupyter Notebook is running
checkpoint = torch.load('VAE_LLExtraction.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

VariationalAutoencoder(
  (var1_emb): Embedding(4, 8)
  (var2_emb): Embedding(60, 8)
  (var3_emb): Embedding(41, 8)
  (encoder_fc): Sequential(
    (0): Linear(in_features=24, out_features=16, bias=True)
    (1): ReLU()
  )
  (fc_mu): Linear(in_features=16, out_features=8, bias=True)
  (fc_logvar): Linear(in_features=16, out_features=8, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=24, bias=True)
  )
  (var1_out): Linear(in_features=8, out_features=4, bias=True)
  (var2_out): Linear(in_features=8, out_features=60, bias=True)
  (var3_out): Linear(in_features=8, out_features=41, bias=True)
)

Test if initial data can be reconstructed with accuracy and get bound for Z dimension.

In [4]:
# Get bounds of continuous latent space
z , min_bounds , max_bounds = vae_model.get_z_dimension(initial_points, model)

Test if initial data can be reconstructed with accuracy and get bound for Z dimension.

In [5]:
# Get bounds of continuous latent space
z , min_bounds , max_bounds = vae_model.get_z_dimension(initial_points, model)

### Begin Bayesian Optimization algorithm

Generate points to evaluate with ASPEN, the bounds must correspond to the same as the autoencoder and decoder, otherwise the latent space can be projected with high uncertainty if these are recommended by B.O.

In [ ]:
# Manipulated variables 
#                       Re, S/F, SolventType, F, NT 
bounds = torch.tensor([[0.1, 1.5 , 0 , 1 , 20] ,
                       [3.0, 10.0, 3, 60, 60]],dtype = torch.float64)
var_types = ['real', 'real', 'int', 'int', 'int']

# Get initial points
num_initial_points = 10
initial_points_BO = vae_model.population_generation(bounds, var_types , 
                                          num_initial_points=num_initial_points)
print('Number of points generated:', len(initial_points_BO))
# initial_points_BO

Number of points generated: 10


In [ ]:
# Discrete variables, format must be changed to long or int type
continuos_initial_points = initial_points_BO[:, 0:2].numpy()
discrete_initial_points = torch.tensor(np.array(initial_points_BO[:, 2:]),dtype=torch.long)
converted_points = vae_model.transform_discrete(continuos_initial_points,
                             discrete_initial_points, model)


In [ ]:
Application = vae_model.AspenSimulation_Reboiler(aspen_file_path1= "CaprylicAcid_ST1.bkp" , aspen_file_path2 = "CaprylicAcid_ST2.bkp" )
Application.connect()
initial_evals = torch.tensor(np.array([Application.run_simulation(point) for point in initial_points_BO]), dtype=torch.float64)
print(initial_evals.shape)
print('Max function value:', initial_evals[:,0].max().item())
print('Min function value:', initial_evals[:,0].min().item())
Application.close()

torch.Size([10, 1])
Max function value: -54.093126080160005
Min function value: -593.3044695384


In [ ]:
# ELIMINAR
class SingleObjectiveBayesianOpt:
    """
    A method for single-objective optimization problems.

    This class find optimal solutions when a single objectives exist. It automatically handles:
    - Gaussian Process model training for each objective
    - Selection of uncertain points for iterative improvement
    - Multiple optimization iterations with automatic data augmentation

    Attributes:
        X_init (torch tensor): Initial decision variable values (unscaled)
        Y_init (torch tensor): Initial objective function values (unscaled)
        var_names (list): Names of decision variables
        obj_names (list): Names of objective functions
        n_iterations (int): Number of optimization iterations
        grid_points (int): Number of points in prediction grid
        X_ranges (dict): Ranges for each decision variable
        X_scaled (np.array): Scaled decision variables (-1 to 1)
        X_real (np.array): Unscaled decision variables
        Y (np.array): Objective function values
        gpr_models (list): Trained Gaussian Process models
        history (list): Optimization history
        predictions (np.array): Grid predictions
        std_devs (np.array): Prediction standard deviations
    """

    def __init__(self, X_init, Y_init,  bounds, model_enddec, 
                 obj_names, latent_dim, n_evals, method, Aspen_Application):
        """
        Args:
            X_init (tensor): Initial decision variable values (unscaled)
            Y_init (tensor): Initial objective function values (unscaled)
            obj_names (list): Names of objective functions
            latent_dim (int): Dimention of the latent space
            n_evals (int): Number of evaluations that will be run 
            method (str): Define which method will be used, it can be 
            specified as upc, ei
        """

        # Initialize Aspen
        self.Aspen_Application = Aspen_Application
        self.Aspen_Application.connect()

        # Additional variables 
        self.n_evals = n_evals
        self.X_init = X_init
        self.all_points = X_init
        self.Y_init = Y_init
        self.all_objectives = Y_init

        self.obj_names = obj_names
        self.bounds = bounds
        self.model_enddec = model_enddec
        self.latent_dim = latent_dim

        # Training data
        self.method = method
        # Prediction results
        self.predictions = None
        self.std_devs = None

    def train_gpr_model(self, X, Y ):
        """Train independent Gaussian Process models for each objective function."""
        self.gp_models = []
        for i in range(Y.shape[1]):
            gp = SingleTaskGP(X, Y[:,i].unsqueeze(1) , 
                                input_transform=Normalize(d=10),
                                outcome_transform=Standardize(m=1))
            self.gp_models.append(gp)
        self.mlls = [ExactMarginalLogLikelihood(m.likelihood, m) for m in self.gp_models]
        for mll in self.mlls:
            fit_gpytorch_mll(mll)
    
    def prediction_grid_std(self , acqfs,  batch_size = 1000,  n_samples = 200, ktop = 10):
        """
        Evalute the aqcs with the bounds.
        Return: Best points
        """
        n_samples = 5000
        n_dims = bounds.shape[1] 

        # Sample uniform random points in each dimension of the bounds
        random_samples = torch.rand(n_samples, n_dims, dtype=torch.float64)
        lower_bounds = bounds[0]
        upper_bounds = bounds[1]
        scaled_samples = lower_bounds + (upper_bounds - lower_bounds) * random_samples
        self.final_points =  scaled_samples 
        final_points_batched = self.final_points

        batch_size = 100
        acq_values_list = []
        all_indices = []
        offset = 0
        with torch.no_grad():
            for acqf in acqfs:
                local_acq_values  = []
                for i in range(0, final_points_batched.shape[0], batch_size):
                    batch = final_points_batched[i:i + batch_size]  # slice batch
                    acq_vals = acqf(batch.unsqueeze(1))
                    local_acq_values.append(acq_vals)

                # Save values ​​for each pothole for each aqf
                local_acq_values = torch.cat(local_acq_values, dim=0)
                # Save the values ​​in a global list
                acq_values_list.append(local_acq_values)


                # Save all indexes for each aqf
                indices = torch.arange(final_points_batched.shape[0], dtype=torch.long) + offset
                all_indices.append( indices)
                # Update offset to keep indexes different
                offset += final_points_batched.shape[0]
        
        acq_values = torch.cat(acq_values_list, dim=0)
        total_indices = torch.cat(all_indices, dim=0)
        
        #  Get top K candidates by acquisition
        topk_vals, topk_indices = torch.topk(acq_values.squeeze(-1), k=ktop)
        topk_final_indices = total_indices[topk_indices]
        # Get the best points according to the aqf to test with GPS
        final_points_expanded = final_points_batched.repeat(len(acqfs), 1)
        topk_candidates = final_points_batched[topk_final_indices]

        return  topk_candidates

    def converted_candidates(self):
        # Pass z through the decoder
        z_space_predicted = torch.tensor([self.all_points[:, -self.latent_dim:].numpy()] , dtype=torch.float32)
        f1_discrete, f2_discrete, st_discrete = self.model_enddec.recover(z_space_predicted)
        return torch.cat((self.all_points[:,:2], f1_discrete.unsqueeze(1), f2_discrete.unsqueeze(1), st_discrete.unsqueeze(1)), dim=1).numpy()

    def run_optimization(self):
        """
        Run the complete optimization process for all iterations.

        """
        # Listas para salvar datos
        self.latent_space_BO = []
        self.tracking_OF = []
        self.utopia_distance = []
        self.all_points, self.all_objectives = self.X_init, self.Y_init
        number_eval = self.X_init.shape[0]

        print('Start optimization process')
        while number_eval < self.n_evals:
            if self.method == 'UCB':
                self.train_gpr_model(self.all_points, self.all_objectives)
                UpperConfidenceBound_acquisitions = [UpperConfidenceBound(gp, beta=0.1) for gp in self.gp_models]
                best_candidates = self.prediction_grid_std(UpperConfidenceBound_acquisitions)
                optimal_candidates = best_candidates
                
                # Pass z through the decoder
                z_space_predicted = torch.tensor(optimal_candidates[:, 2:].numpy(), dtype=torch.float32).unsqueeze(0)
                self.latent_space_BO.append(z_space_predicted)

                f1_discrete, f2_discrete, st_discrete = self.model_enddec.recover(z_space_predicted)

                new_candidates  = torch.cat((optimal_candidates[:,0:2], f1_discrete.unsqueeze(1), 
                                            f2_discrete.unsqueeze(1), st_discrete.unsqueeze(1)), dim=1).numpy()

                new_eval  = torch.tensor(np.array([self.Aspen_Application.run_simulation(point) for point in new_candidates]), dtype=torch.float64)

                # Update data collected by the GP

                self.all_points = torch.cat([self.all_points, optimal_candidates ], dim = 0 )  
                self.all_objectives = torch.cat([self.all_objectives, new_eval], dim = 0  ) 

                self.tracking_OF.append(self.all_objectives.max().item())
                number_eval = self.all_points.shape[0]
                print('number of functions evaluations:', number_eval)
            elif self.method == 'EI':
                self.train_gpr_model(self.all_points, self.all_objectives)
                ExpectedImprovement_acquisitions  = [ExpectedImprovement(gp,best_f = self.all_objectives.max().item()  ) for i, gp in enumerate(self.gp_models)]
                best_candidates = self.prediction_grid_std(ExpectedImprovement_acquisitions)
                optimal_candidates = best_candidates

                # Pass z through the decoder
                z_space_predicted = torch.tensor(optimal_candidates[:, 2:].numpy(), dtype=torch.float32).unsqueeze(0)
                self.latent_space_BO.append(z_space_predicted)

                f1_discrete, f2_discrete, st_discrete = self.model_enddec.recover(z_space_predicted)

                new_candidates  = torch.cat((optimal_candidates[:,0:2], f1_discrete.unsqueeze(1), 
                                            f2_discrete.unsqueeze(1), st_discrete.unsqueeze(1)), dim=1).numpy()

                new_eval  = torch.tensor(np.array([self.Aspen_Application.run_simulation(point) for point in new_candidates]), dtype=torch.float64)

                # Update data collected by the GP

                self.all_points = torch.cat([self.all_points, optimal_candidates ], dim = 0 )  
                self.all_objectives = torch.cat([self.all_objectives, new_eval], dim = 0  ) 

                self.tracking_OF.append(self.all_objectives.max().item())
                number_eval = self.all_points.shape[0]
                print('number of functions evaluations:', number_eval)

        # Close Aspen :D              
        self.Aspen_Application.close()

    def save_data(self, file = r'test.csv'):
        # Objective functions
        df1 = pd.DataFrame(data=self.all_objectives.numpy() , columns=self.obj_names)
        data_names = [ 'x'+ str(i+1)  for i in range(self.converted_candidates().shape[1])]
        # Initial data used to train GP
        df2 = pd.DataFrame(data=self.converted_candidates() , columns= data_names)
        
        
        # Get initial latent space in X_train
        initial_z_space = self.X_init[:, -self.latent_dim:].numpy()
        # Get new latent space explored
        latent_space_BOs = self.latent_space_BO
        latent_BO_matrix = torch.cat(latent_space_BOs, dim=1).numpy().squeeze(0)
        total_latent = np.concatenate((initial_z_space,latent_BO_matrix),axis = 0)
        space_names =  [ 'z'+ str(i+1)  for i in range(total_latent.shape[1])]
        df3 = pd.DataFrame(data= total_latent, columns =space_names )
        df = pd.concat([df1,df2,df3], axis= 1)
        df = df.drop_duplicates()
        df.to_csv(file, index=False)  


Single objective procedure -  B.O

In [50]:
bounds = torch.tensor([[0.1, 0.5, min_bounds[0] , min_bounds[1] , min_bounds[2], min_bounds[3], min_bounds[4], min_bounds[5], min_bounds[6], min_bounds[7]] ,
                       [3.0, 10 , max_bounds[0], max_bounds[1] , max_bounds[2], max_bounds[3], max_bounds[4], max_bounds[5], max_bounds[6], max_bounds[7] ]],dtype = torch.float64)

In [ ]:
# Parameters for VAE
device = 'cuda' if torch.cuda.is_available() else 'cpu'
latent_dim = 8
# Load again model 
model = vae_model.VariationalAutoencoder(emb_dim=8,   latent_dim= latent_dim ).to(device)
checkpoint = torch.load('VAE_LLExtraction.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Create instance for Aspen
Application = vae_model.AspenSimulation_Reboiler(aspen_file_path1= "CaprylicAcid_ST1.bkp" , aspen_file_path2 = "CaprylicAcid_ST2.bkp" )


all_points = converted_points
all_objectives = initial_evals
obj_names = ['reboiler']
z_train = z
# Number of evaluations (Stopping criteria) 
# Initial data were 10 points , so the process end get all data reachs 20 points
n_evals = 500 
bounds = torch.tensor([[0.1, 0.5, min_bounds[0] , min_bounds[1] , min_bounds[2], min_bounds[3], min_bounds[4], min_bounds[5], min_bounds[6], min_bounds[7]] ,
                       [3.0, 10 , max_bounds[0], max_bounds[1] , max_bounds[2], max_bounds[3], max_bounds[4], max_bounds[5], max_bounds[6], max_bounds[7] ]],dtype = torch.float64)
MOBO= SingleObjectiveBayesianOpt( X_init = all_points , Y_init = initial_evals, 
                        bounds = bounds,  model_enddec = model,  
                        obj_names = obj_names , latent_dim = latent_dim,
                        n_evals = n_evals, method='EI', Aspen_Application = Application ) 
                        # n_evals = n_evals, method='UCB', Aspen_Application = Application )  # other option 

MOBO.run_optimization()  

Start optimization process
number of functions evaluations: 20
number of functions evaluations: 30
number of functions evaluations: 40
number of functions evaluations: 50


In [ ]:
# # Just in case something goes wrong
# MOBO.Aspen_Application.close()


In [ ]:
final_points = MOBO.final_points 
final_points_batched = final_points
batch_size = 100

points_gps = []
points_std = []
with torch.no_grad():
    for gp in MOBO.gp_models:
        local_mean_values  = []
        local_std_values = []
        for i in range(0, final_points_batched.shape[0], batch_size):
            batch = final_points_batched[i:i + batch_size]  # slice batch


            prediction = gp.posterior(batch) #batch.unsqueeze(-1))
            local_mean_values.append(prediction.mean)
            local_std_values.append(prediction.stddev.unsqueeze(1))
        local_mean_values = torch.cat(local_mean_values, dim=0)
        local_std_values = torch.cat(local_std_values, dim=0)
        points_gps.append(local_mean_values)
        points_std.append(local_std_values)

all_means = torch.cat(points_gps, dim=1).squeeze(-1)
all_std = torch.cat(points_std, dim=1).squeeze(-1)      
last_points = final_points[:,-8:] 
last_means = all_means  
last_stds = all_std          

In [ ]:
vae_model.plot_tsne_reboiler(last_points, last_means, 
                           last_stds, file = 'Reboiler.eps' )

In [ ]:
# Save data for further analysis
# MOBO.save_data(file = r'LLExtraction_REB.csv' )

C:\Users\gabo1\AppData\Local\Temp\ipykernel_23084\439050618.py:267: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ..\torch\csrc\utils\tensor_new.cpp:278.)
  z_space_predicted = torch.tensor([self.all_points[:, -self.latent_dim:].numpy()] , dtype=torch.float32)
